# Uvod u Z3 kroz logiku prvog reda

Ova sveska uvodi Z3 pre svega kao alat za modelovanje i proveru formula logike prvog reda.

Cilj nije da pišemo algoritam koji rešava problem, nego da opišemo uslove koje svet/model mora da zadovolji.

Centralna ideja:

> U Z3 ne govorimo kako da se rešenje pronađe, nego opisujemo kako rešenje mora da izgleda.

Koristićemo:
- konstante,
- predikate,
- funkcije,
- kvantifikatore,
- modele,
- dokazivanje preko negacije zaključka.

## 1. Prvi kontakt sa Z3

Z3 radi sa simboličkim izrazima.

Na primer, kada napišemo:

```python
x = Int('x')
```

ne pravimo običnu Python promenljivu sa konkretnom vrednošću, nego simboličku promenljivu nad kojom Z3 rezonuje.

In [22]:
from z3 import *

x = Int('x')
y = Int('y')

s = Solver()

s.add(x > 0)
s.add(y == x + 2)

print(s.check())
print(s.model())

sat
[y = 3, x = 1]


`Solver` čuva ograničenja.

```python
s.add(...)
```

dodaje nove uslove.

```python
s.check()
```

pita da li postoji model koji zadovoljava sva ograničenja.

Mogući odgovori su:

- `sat` — postoji model,
- `unsat` — ne postoji model,
- `unknown` — Z3 ne zna da odluči.

## 2. Model nije dokaz jedinstvenosti

Ako Z3 vrati `sat`, on daje jedan model, ne nužno jedini model.

In [23]:
x = Int('x')
y = Int('y')

s = Solver()

s.add(x + y == 10)
s.add(x > y)

print(s.check())

m = s.model()
print(m)

print("x =", m.evaluate(x))
print("y =", m.evaluate(y))
print("x + y =", m.evaluate(x + y))

sat
[y = 4, x = 6]
x = 6
y = 4
x + y = 10


Važna poenta:

> `model()` daje jedan konkretan svet u kome su uslovi tačni.

To ne znači da je to jedino rešenje.

## 3. Osnovni pojmovi logike prvog reda u Z3

U logici prvog reda imamo:

- domene,
- konstante,
- predikate,
- funkcije,
- kvantifikatore.

U Z3 ih modelujemo ovako.

In [24]:
Osoba = DeclareSort('Osoba')

Sokrat = Const('Sokrat', Osoba)

Covek = Function('Covek', Osoba, BoolSort())
Smrtan = Function('Smrtan', Osoba, BoolSort())

Otac = Function('Otac', Osoba, Osoba)

x = Const('x', Osoba)

Objašnjenje:

```python
Osoba = DeclareSort('Osoba')
```

uvodi domen objekata.

```python
Sokrat = Const('Sokrat', Osoba)
```

uvodi jedan konkretan objekat iz tog domena.

```python
Covek = Function('Covek', Osoba, BoolSort())
```

uvodi predikat:

$$
Covek : Osoba \to Bool
$$

```python
Otac = Function('Otac', Osoba, Osoba)
```

uvodi funkciju:

$$
Otac : Osoba \to Osoba
$$

Predikat vraća tačno/netačno.

Funkcija vraća objekat.

## 4. Klasičan primer: Sokrat

Premise:

1. Svi ljudi su smrtni.
2. Sokrat je čovek.

Zaključak:

3. Sokrat je smrtan.

In [25]:
from z3 import *

Osoba = DeclareSort('Osoba')

Sokrat = Const('Sokrat', Osoba)

Covek = Function('Covek', Osoba, BoolSort())
Smrtan = Function('Smrtan', Osoba, BoolSort())

x = Const('x', Osoba)

s = Solver()

# Svi ljudi su smrtni.
s.add(
    ForAll(x, Implies(Covek(x), Smrtan(x)))
)

# Sokrat je čovek.
s.add(Covek(Sokrat))

# Proveravamo da li zaključak mora da važi.
# Dodajemo negaciju zaključka.
s.add(Not(Smrtan(Sokrat)))

print(s.check())

unsat


Očekujemo:

```text
unsat
```

Zašto?

Zato što ne postoji model u kome su obe premise tačne, a zaključak lažan.

Drugim rečima:

$$
Premise \land \neg Zaključak
$$

je nezadovoljivo.

Zato:

$$
Premise \Rightarrow Zaključak
$$

## 5. Zašto dodajemo negaciju zaključka?

Ako direktno dodamo zaključak:

```python
s.add(Smrtan(Sokrat))
```

i dobijemo `sat`, nismo dokazali da zaključak sledi.

Samo smo pokazali da je zaključak kompatibilan sa premisama.

Za dokazivanje želimo da proverimo da li postoji kontra-primer.

In [26]:
from z3 import *

Osoba = DeclareSort('Osoba')

Sokrat = Const('Sokrat', Osoba)

Covek = Function('Covek', Osoba, BoolSort())
Smrtan = Function('Smrtan', Osoba, BoolSort())

x = Const('x', Osoba)

s = Solver()

s.add(ForAll(x, Implies(Covek(x), Smrtan(x))))
s.add(Covek(Sokrat))

# Pogrešan dokazni pokušaj:
s.add(Smrtan(Sokrat))

print(s.check())
print(s.model())

sat
[Sokrat = Osoba!val!0,
 Smrtan = [else -> True],
 Covek = [else -> True]]


Rezultat `sat` ovde ne znači da smo dokazali tvrdnju.

Mi smo zaključak sami dodali među uslove.

Ispravan obrazac za proveru zaključka je:

```python
s.add(premise)
s.add(Not(zakljucak))
print(s.check())
```

Ako je rezultat `unsat`, zaključak logički sledi.

## 6. Primer gde zaključak ne sledi

Premise:

1. Svi studenti koji slušaju veštačku inteligenciju znaju Python.
2. Pera zna Python.

Pitanje:

Da li iz toga sledi da Pera sluša veštačku inteligenciju?

Ne.

In [27]:
from z3 import *

Osoba = DeclareSort('Osoba')

SlusaVI = Function('SlusaVI', Osoba, BoolSort())
ZnaPython = Function('ZnaPython', Osoba, BoolSort())

Pera = Const('Pera', Osoba)
x = Const('x', Osoba)

s = Solver()

s.add(
    ForAll(x, Implies(SlusaVI(x), ZnaPython(x)))
)

s.add(ZnaPython(Pera))

# Pokušavamo da dokažemo da Pera sluša VI.
# Dodajemo negaciju zaključka.
s.add(Not(SlusaVI(Pera)))

print(s.check())
print(s.model())

sat
[Pera = Osoba!val!0,
 ZnaPython = [else -> True],
 SlusaVI = [else -> False]]


Dobijamo `sat`.

To znači da postoji kontra-primer:

- Pera zna Python,
- ali ne sluša veštačku inteligenciju.

Ovo je tipična logička greška:

$$
A \Rightarrow B
$$

$$
B
$$

ne implicira:

$$
A
$$

## 7. Modus ponens

Sada primer gde zaključak zaista sledi.

Premise:

1. Svi studenti koji slušaju VI rade domaći.
2. Pera sluša VI.

Zaključak:

3. Pera radi domaći.

In [28]:
from z3 import *

Osoba = DeclareSort('Osoba')

SlusaVI = Function('SlusaVI', Osoba, BoolSort())
RadiDomaci = Function('RadiDomaci', Osoba, BoolSort())

Pera = Const('Pera', Osoba)
x = Const('x', Osoba)

s = Solver()

s.add(
    ForAll(x, Implies(SlusaVI(x), RadiDomaci(x)))
)

s.add(SlusaVI(Pera))

# Negacija zaključka
s.add(Not(RadiDomaci(Pera)))

print(s.check())

unsat


Rezultat je `unsat`.

Ne postoji model u kome:

- svako ko sluša VI radi domaći,
- Pera sluša VI,
- Pera ne radi domaći.

Dakle zaključak važi.

## 8. Funkcije u logici prvog reda

Predikat vraća `Bool`.

Funkcija vraća objekat.

Primer:

```python
Majka = Function('Majka', Osoba, Osoba)
```

Ovo znači:

$$
Majka : Osoba \to Osoba
$$

Dakle, za svaku osobu funkcija `Majka` vraća neku osobu.

Važno:

> Funkcije u logici prvog reda su totalne.

To znači da svaka vrednost iz domena mora imati rezultat.

## 9. Relacije sa više argumenata

Predikati mogu imati više argumenata.

Na primer:

```python
Roditelj(x, y)
```

može značiti:

> x je roditelj osobe y.

## 10. Roditelj i predak

Premise:

1. Ako je x roditelj y, onda je x predak y.
2. Ako je x roditelj y i y roditelj z, onda je x predak z.
3. Marko je roditelj Ane.
4. Ana je roditelj Jovana.

Zaključak:

5. Marko je predak Jovana.

In [30]:
from z3 import *

Osoba = DeclareSort('Osoba')

Marko = Const('Marko', Osoba)
Ana = Const('Ana', Osoba)
Jovan = Const('Jovan', Osoba)

Roditelj = Function('Roditelj', Osoba, Osoba, BoolSort())
Predak = Function('Predak', Osoba, Osoba, BoolSort())

x, y, z = Consts('x y z', Osoba)

s = Solver()

s.add(
    ForAll([x, y],
        Implies(Roditelj(x, y), Predak(x, y))
    )
)

s.add(
    ForAll([x, y, z],
        Implies(
            And(Roditelj(x, y), Roditelj(y, z)),
            Predak(x, z)
        )
    )
)

s.add(Roditelj(Marko, Ana))
s.add(Roditelj(Ana, Jovan))

# Negacija zaključka
s.add(Not(Predak(Marko, Jovan)))

print(s.check())

unsat


Rezultat je `unsat`.

Ovo nije potpuna definicija pretka za proizvoljnu dubinu stabla, ali je dobar prvi primer relacije sa dva argumenta i pravila sa tri promenljive.

## 11. Primer sa braćom

Premise:

1. Ako su dve osobe braća, imaju zajedničkog roditelja.
2. Roditelj je stariji od deteta.
3. Postoje dve osobe koje su braća.

Zaključak:

4. Postoji neko ko je stariji od nekoga.

In [31]:
from z3 import *

Osoba = DeclareSort('Osoba')

Braca = Function('Braca', Osoba, Osoba, BoolSort())
Roditelj = Function('Roditelj', Osoba, Osoba, BoolSort())
Stariji = Function('Stariji', Osoba, Osoba, BoolSort())

x, y, z = Consts('x y z', Osoba)

s = Solver()

# Ako su x i y braća, postoji z koji je roditelj obojici.
s.add(
    ForAll([x, y],
        Implies(
            Braca(x, y),
            Exists(z, And(Roditelj(z, x), Roditelj(z, y)))
        )
    )
)

# Roditelj je stariji od deteta.
s.add(
    ForAll([x, y],
        Implies(Roditelj(x, y), Stariji(x, y))
    )
)

# Postoje braća.
s.add(
    Exists([x, y], Braca(x, y))
)

# Zaključak: postoji neko ko je stariji od nekoga.
zakljucak = Exists([x, y], Stariji(x, y))

# Negacija zaključka
s.add(Not(zakljucak))

print(s.check())

unsat


Rezultat je `unsat`.

Zašto?

Ako postoje braća, onda imaju zajedničkog roditelja.

Ako je neko roditelj, onda je stariji od deteta.

Dakle mora postojati par osoba takav da je jedna starija od druge.

## 12. Pažnja: mesto kvantifikatora je važno

Ove dve formule nisu iste:

```python
ForAll([x, y], Implies(Braca(x, y), Exists(z, Roditelj(z, x))))
```

i

```python
ForAll([x, y], Exists(z, Implies(Braca(x, y), Roditelj(z, x))))
```

Prva znači:

> Ako su x i y braća, onda postoji roditelj.

Druga znači:

> Za svaka x i y postoji z takav da važi implikacija.

Druga je često slabija nego što želimo, jer ako x i y nisu braća, implikacija je automatski tačna.

## 13. Geometrijski primer sa pravama

Premise:

1. Ako su dve prave nemimoilazne, onda se seku ili su paralelne.
2. Ako se dve prave seku, onda su u istoj ravni.
3. Ako su dve prave paralelne, onda su u istoj ravni.

Zaključak:

4. Ako su dve prave nemimoilazne, onda su u istoj ravni.

In [32]:
from z3 import *

Prava = DeclareSort('Prava')

Nemimoilazne = Function('Nemimoilazne', Prava, Prava, BoolSort())
SekuSe = Function('SekuSe', Prava, Prava, BoolSort())
Paralelne = Function('Paralelne', Prava, Prava, BoolSort())
IstaRavan = Function('IstaRavan', Prava, Prava, BoolSort())

x, y = Consts('x y', Prava)

s = Solver()

s.add(
    ForAll([x, y],
        Implies(
            Nemimoilazne(x, y),
            Or(SekuSe(x, y), Paralelne(x, y))
        )
    )
)

s.add(
    ForAll([x, y],
        Implies(SekuSe(x, y), IstaRavan(x, y))
    )
)

s.add(
    ForAll([x, y],
        Implies(Paralelne(x, y), IstaRavan(x, y))
    )
)

zakljucak = ForAll([x, y],
    Implies(Nemimoilazne(x, y), IstaRavan(x, y))
)

s.add(Not(zakljucak))

print(s.check())

unsat


Rezultat je `unsat`.

Dokazna struktura:

Ako su prave nemimoilazne, onda važi bar jedan od dva slučaja:

- seku se,
- paralelne su.

U oba slučaja su u istoj ravni.

Dakle moraju biti u istoj ravni.

## 14. Primer sa gradovima i državama

Premise:

1. Svaki grad pripada nekoj državi.
2. Ako grad pripada državi, onda je ta država povezana sa tim gradom.
3. Beograd je grad.

Zaključak:

4. Postoji država koja je povezana sa Beogradom.

In [33]:
from z3 import *

Objekat = DeclareSort('Objekat')

Grad = Function('Grad', Objekat, BoolSort())
Drzava = Function('Drzava', Objekat, BoolSort())
Pripada = Function('Pripada', Objekat, Objekat, BoolSort())
PovezanaSa = Function('PovezanaSa', Objekat, Objekat, BoolSort())

Beograd = Const('Beograd', Objekat)

g, d = Consts('g d', Objekat)

s = Solver()

# Svaki grad pripada nekoj državi.
s.add(
    ForAll(g,
        Implies(
            Grad(g),
            Exists(d, And(Drzava(d), Pripada(g, d)))
        )
    )
)

# Ako grad pripada državi, država je povezana sa gradom.
s.add(
    ForAll([g, d],
        Implies(
            And(Grad(g), Drzava(d), Pripada(g, d)),
            PovezanaSa(d, g)
        )
    )
)

s.add(Grad(Beograd))

zakljucak = Exists(d, And(Drzava(d), PovezanaSa(d, Beograd)))

s.add(Not(zakljucak))

print(s.check())

unsat


Rezultat je `unsat`.

Ovaj primer je dobar jer kombinuje:

- univerzalni kvantifikator,
- egzistencijalni kvantifikator,
- binarne relacije,
- dokazivanje egzistencijalnog zaključka.

## 15. Primer gde Z3 nalazi kontra-primer

Premise:

1. Svaki grad pripada nekoj državi.
2. Beograd je grad.

Pogrešan zaključak:

3. Beograd pripada Srbiji.

Ovo ne sledi ako nigde nismo rekli da je Srbija ta država.

In [34]:
from z3 import *

Objekat = DeclareSort('Objekat')

Grad = Function('Grad', Objekat, BoolSort())
Drzava = Function('Drzava', Objekat, BoolSort())
Pripada = Function('Pripada', Objekat, Objekat, BoolSort())

Beograd = Const('Beograd', Objekat)
Srbija = Const('Srbija', Objekat)

g, d = Consts('g d', Objekat)

s = Solver()

s.add(
    ForAll(g,
        Implies(
            Grad(g),
            Exists(d, And(Drzava(d), Pripada(g, d)))
        )
    )
)

s.add(Grad(Beograd))
s.add(Drzava(Srbija))

# Pokušavamo da dokažemo da Beograd pripada Srbiji.
zakljucak = Pripada(Beograd, Srbija)

# Dodajemo negaciju zaključka.
s.add(Not(zakljucak))

print(s.check())
print(s.model())

sat
[Beograd = Objekat!val!0,
 Srbija = Objekat!val!1,
 Grad = [else -> True],
 Drzava = [else -> True],
 Pripada = [else -> Var(1) == Objekat!val!2]]


Rezultat je `sat`.

To znači da postoji model u kome:

- Beograd jeste grad,
- Srbija jeste država,
- svaki grad pripada nekoj državi,
- ali Beograd ne pripada Srbiji.

Z3 time pokazuje da zaključak ne sledi iz datih premisa.

Da bi zaključak sledio, morali bismo dodati još neku premisu, na primer:

```python
s.add(Pripada(Beograd, Srbija))
```

ili pravilo koje jednoznačno određuje državu kojoj Beograd pripada.

## 16. Injektivnost funkcije

Možemo izraziti da je funkcija injektivna.

Za funkciju:

$$
f : A \to A
$$

injektivnost znači:

$$
f(x) = f(y) \Rightarrow x = y
$$

In [36]:
from z3 import *

A = DeclareSort('A')

f = Function('f', A, A)

x, y = Consts('x y', A)

s = Solver()

# f je injektivna
s.add(
    ForAll([x, y],
        Implies(f(x) == f(y), x == y)
    )
)

# Pokušavamo da nađemo kontra-primer injektivnosti
a, b = Consts('a b', A)

s.add(a != b)
s.add(f(a) == f(b))

print(s.check())

unsat


Rezultat je `unsat`.

To je očekivano, jer smo prvo rekli da je funkcija injektivna, a zatim pokušali da pronađemo dva različita elementa sa istom slikom.

In [37]:
# Rešenje

from z3 import *

Osoba = DeclareSort('Osoba')

Asistent = Function('Asistent', Osoba, BoolSort())
DrziVezbe = Function('DrziVezbe', Osoba, BoolSort())
RadiSaStudentima = Function('RadiSaStudentima', Osoba, BoolSort())

Denis = Const('Denis', Osoba)
x = Const('x', Osoba)

s = Solver()

s.add(
    ForAll(x, Implies(Asistent(x), DrziVezbe(x)))
)

s.add(
    ForAll(x, Implies(DrziVezbe(x), RadiSaStudentima(x)))
)

s.add(Asistent(Denis))

zakljucak = RadiSaStudentima(Denis)

s.add(Not(zakljucak))

print(s.check())

unsat


## 17. Završni obrazac

Za proveru da li zaključak sledi iz premisa koristimo obrazac:

```python
s = Solver()

s.add(premisa_1)
s.add(premisa_2)
...
s.add(premisa_n)

s.add(Not(zakljucak))

print(s.check())
```

Ako dobijemo:

```text
unsat
```

zaključak sledi.

Ako dobijemo:

```text
sat
```

zaključak ne sledi; Z3 je našao kontra-primer.

Glavna ideja:

> Z3 ne proverava da li je zaključak lep, intuitivan ili verovatan. Proverava da li logički mora da važi iz zadatih premisa.